In [87]:
import pandas as pd
import numpy as np

import sys
sys.path.append('Level-Set-Boosting')
import LSBoost
from helper_functions import MSCE as MSCE

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score # Calculate the accuracy
#from imblearn.over_sampling import RandomOverSampler
from sklearn import metrics as sm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.calibration import CalibratedClassifierCV

import matplotlib.pyplot as plt
import seaborn as sns

from utils import *
from KMultiAcc import KMultiAcc
from MCBoost import MCBoost
from folktables import ACSDataSource, ACSEmployment, ACSIncome, ACSPublicCoverage, ACSMobility

In [88]:
# Health Public Coverage Task WI
data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')
data = data_source.get_data(states= ["WI"], download=True)
features, labels, _ = ACSPublicCoverage.df_to_numpy(data)
prefix = "Health_WI"

### Train Baseline and Post-processed Models

In [89]:
baseline_model="Logistic_Regression"
witness_metric = 'rbf'
seed = 42

In [90]:
scaler = StandardScaler()
#fit transform data
features = scaler.fit_transform(features, labels)
labels = labels.astype('int')

X_train_wit_val, X_test, y_train_wit_val, y_test = train_test_split(
    features, labels, test_size=0.3, random_state=seed)

X_train, X_wit_val, y_train, y_wit_val = train_test_split(
    X_train_wit_val, y_train_wit_val, test_size=0.4, random_state=seed)

X_wit, X_val, y_wit, y_val = train_test_split(X_wit_val, y_wit_val, test_size=0.4, random_state=seed)

if baseline_model == "Logistic_Regression":
    model = LogisticRegression(max_iter=10000)
    weak_learner = LogisticRegression(max_iter=10000)
elif baseline_model == "Decision_Tree":
    model = DecisionTreeClassifier(max_depth = 1)
    weak_learner = DecisionTreeClassifier(max_depth = 1)
elif baseline_model == "Random_Forest":
    model = RandomForestClassifier(max_depth=2, random_state=seed)
    weak_learner = RandomForestClassifier(max_depth=2, random_state=seed)
else:
    print("Baseline not supported")

# fit baseline model
model.fit(X_train, y_train)

# fit KMAcc 
kma = KMultiAcc(baseline_model = baseline_model)
kma.fit_model(X_train, y_train)
kma.fit(X_wit_val, y_wit_val, X_wit, y_wit, X_val, y_val)

# kMAcc + isotonic calibration
kMAcc_iso = CalibratedClassifierCV(kma, cv="prefit", method="isotonic")
kMAcc_iso.fit(X_val, y_val)

# baseline + isotonic calibration
base_iso = CalibratedClassifierCV(model, cv="prefit", method="isotonic")
base_iso.fit(X_val, y_val)

#LSBoost
LSBoostReg = LSBoost.LSBoostingRegressor(
                                T = 100,
                                num_bins = 100,
                                min_group_size = 5,
                                global_gamma = .005,
                                weak_learner=weak_learner,
                                bin_type = 'distribution',
                                learning_rate = .1,
                                initial_model = None,
                                final_round = True,
                                center_mean=False)
LSBoostReg.fit(X_train_wit_val, y_train_wit_val)

# MCBoost
f_MCBoost = MCBoost(partition = True, multiplicative = True, init_predictor = gen_preds(model), max_iter = 10)
f_MCBoost.fit(X_train_wit_val, y_train_wit_val)

# MCBoost + isotonic calibration
mcboost_iso = CalibratedClassifierCV(f_MCBoost, cv="prefit", method="isotonic")
mcboost_iso.fit(X_val, y_val)

Optimal Gamma: 1.5
Optimal lambda: 0.062


LS:   0%|           0/100 ?, Training Error=0.2099762

Early Termination at round: 2
Memory released!


CalibratedClassifierCV(cv='prefit',
                       estimator=<MCBoost.MCBoost object at 0x28cbfafe0>,
                       method='isotonic')

In [91]:
# Predictions of various models
baseline_test_prob = model.predict_proba(X_test)[:,1].reshape(len(X_test),1)
mcb_test_prob = f_MCBoost.predict_proba(X_test)[:,1].reshape(len(X_test),1)
kma_test_prob = kma.predict_proba(X_test)[:,1].reshape(len(X_test),1)
kma_iso_test_prob = kMAcc_iso.predict_proba(X_test)[:,1].reshape(len(X_test),1)
base_iso_test_prob = base_iso.predict_proba(X_test)[:,1].reshape(len(X_test),1)
lsboost_test_prob = LSBoostReg.predict_proba(X_test)[:,1].reshape(len(X_test),1)
mcboost_iso_test_prob = mcboost_iso.predict_proba(X_test)[:,1].reshape(len(X_test),1)


### Analyze Pre-defined Group Performance

In [92]:
features_df, labels_df, _ = ACSPublicCoverage.df_to_pandas(data)
seed = 42

# mappings derived from PUMS documentation https://www.census.gov/programs-surveys/acs/microdata/documentation.html
# binarize race 1: white, 0: non-white
features_df['RAC1P'] = features_df['RAC1P'].apply(lambda x: "White" if x == 1 else "Non-White")
features_df['SEX'] = features_df['SEX'].apply(lambda x: "Male" if x == 1 else "Female")
features_df['NATIVITY'] = features_df['NATIVITY'].apply(lambda x: "Native" if x == 1 else "Foreign Born")

_, X_test, _, y_test = train_test_split(
    features_df, labels_df, test_size=0.3, random_state=seed)

#### Classification Errors

In [93]:
# Group attributes
group_attrs = ['NATIVITY', 'SEX','RAC1P']

# Test dataset with group attributes and error
test_df = pd.DataFrame(X_test, columns=features_df.columns)
test_df = test_df[group_attrs]

# Add a column of 1's dummy to calculate number of samples
test_df["Number of Samples"] = 1

# Add columns of testing error on each sample
baseline_test=(baseline_test_prob>=0.5)
mcb_test=(mcb_test_prob>=0.5)
kma_test=(kma_test_prob>=0.5)
kma_iso_test=(kma_iso_test_prob>=0.5)
lsboost_test=(lsboost_test_prob>=0.5)
mcboost_iso_test=(mcboost_iso_test_prob>=0.5)

test_df["Baseline"] = baseline_test^y_test
test_df["MC Boost"] = mcb_test^y_test
test_df["KMAcc"] = kma_test^y_test
test_df["KMAcc+ISO"] = kma_iso_test^y_test
test_df["LS Boost"] = lsboost_test^y_test

In [94]:
comp_table_test = test_df.groupby(group_attrs).sum()

In [95]:
comp_table_test["Baseline"] /= comp_table_test["Number of Samples"] 
comp_table_test["MC Boost"] /= comp_table_test["Number of Samples"]
comp_table_test["KMAcc"] /= comp_table_test["Number of Samples"]
comp_table_test["KMAcc+ISO"] /= comp_table_test["Number of Samples"]
comp_table_test["LS Boost"] /= comp_table_test["Number of Samples"]

# compute delta error against baseline
comp_table_test["MC Boost"] -= comp_table_test["Baseline"]
comp_table_test["KMAcc"] -= comp_table_test["Baseline"]
comp_table_test["KMAcc+ISO"] -= comp_table_test["Baseline"]
comp_table_test["LS Boost"] -= comp_table_test["Baseline"]

# Add overall performance on the last row
comp_table_test.loc["Total"] = [comp_table_test['Number of Samples'].sum(), test_df["Baseline"].mean(), test_df["MC Boost"].mean() - test_df["Baseline"].mean(),
                                         test_df["KMAcc"].mean() - test_df["Baseline"].mean() ,test_df["KMAcc+ISO"].mean() - test_df["Baseline"].mean(), test_df["LS Boost"].mean() - test_df["Baseline"].mean()]


comp_table_test["Number of Samples"] = comp_table_test["Number of Samples"].astype(int)
columns = comp_table_test.columns.drop(["Number of Samples"])
comp_table_test[columns] = (comp_table_test[columns]*100).round(2)


# rename columns to Delta Error
comp_table_test = comp_table_test.rename(columns={"Baseline": "Baseline Error Rate", "MC Boost": u"MC Boost \u0394", 
                                                  "KMAcc": u"KMAcc \u0394", "KMAcc+ISO": u"KMAcc+ISO \u0394", 
                                                  "LS Boost": u"LS Boost \u0394"})


In [96]:
pd.options.display.multi_sparse = False

In [97]:
comp_table_test

,Number of Samples,Baseline Error Rate,MC Boost Δ,KMAcc Δ,KMAcc+ISO Δ,LS Boost Δ
"(Foreign Born, Female, Non-White)",819,19.05,-1.95,0.00,-0.24,-0.61
"(Foreign Born, Female, White)",484,24.79,-1.86,0.21,0.41,-0.83
"(Foreign Born, Male, Non-White)",411,22.63,0.00,0.00,-0.24,-0.73
"(Foreign Born, Male, White)",314,25.16,-1.27,0.00,-0.32,-0.32
"(Native, Female, Non-White)",805,28.32,0.62,-0.25,-0.25,0.37
"(Native, Female, White)",2672,16.58,-0.07,-0.49,-0.37,-0.04
"(Native, Male, Non-White)",821,29.96,-0.37,-1.83,-1.71,-0.24
"(Native, Male, White)",1957,19.26,0.05,-0.51,-0.15,-0.46
Total,8283,21.03,-0.34,-0.47,-0.37,-0.27


In [98]:
print("Classification Errors on Test Data")
# apply color to the table, darker color means lower value, only to last 4 columns
vmin = comp_table_test[comp_table_test.columns[2:]].min().min()
vmax = 0
columns = comp_table_test.columns.drop(["Number of Samples"])
comp_table_test.style.format(precision=2).background_gradient(cmap='Greens_r', subset=comp_table_test.columns[2:], axis = 1, vmin = vmin, vmax = vmax)


Classification Errors on Test Data


,Number of Samples,Baseline Error Rate,MC Boost Δ,KMAcc Δ,KMAcc+ISO Δ,LS Boost Δ
"('Foreign Born', 'Female', 'Non-White')",819,19.05,-1.95,0.00,-0.24,-0.61
"('Foreign Born', 'Female', 'White')",484,24.79,-1.86,0.21,0.41,-0.83
"('Foreign Born', 'Male', 'Non-White')",411,22.63,0.00,0.00,-0.24,-0.73
"('Foreign Born', 'Male', 'White')",314,25.16,-1.27,0.00,-0.32,-0.32
"('Native', 'Female', 'Non-White')",805,28.32,0.62,-0.25,-0.25,0.37
"('Native', 'Female', 'White')",2672,16.58,-0.07,-0.49,-0.37,-0.04
"('Native', 'Male', 'Non-White')",821,29.96,-0.37,-1.83,-1.71,-0.24
"('Native', 'Male', 'White')",1957,19.26,0.05,-0.51,-0.15,-0.46
Total,8283,21.03,-0.34,-0.47,-0.37,-0.27


#### MSE

In [99]:
# Group attributes
group_attrs = ['NATIVITY', 'SEX','RAC1P']

# Test dataset with group attributes and error
test_df = pd.DataFrame(X_test, columns=features_df.columns)
test_df = test_df[group_attrs]

# Add a column of 1's dummy to calculate number of samples
test_df["Number of Samples"] = 1

test_df["Baseline"] = (baseline_test_prob-y_test)**2
test_df["MC Boost"] = (mcb_test_prob-y_test)**2
test_df["KMAcc"] = (kma_test_prob-y_test)**2
test_df["KMAcc+ISO"] = (kma_iso_test_prob-y_test)**2
test_df["LS Boost"] = (lsboost_test_prob-y_test)**2

In [100]:
comp_table_test = test_df.groupby(group_attrs).sum()

In [101]:
comp_table_test["Baseline"] /= comp_table_test["Number of Samples"] 
comp_table_test["MC Boost"] /= comp_table_test["Number of Samples"]
comp_table_test["KMAcc"] /= comp_table_test["Number of Samples"]
comp_table_test["KMAcc+ISO"] /= comp_table_test["Number of Samples"]
comp_table_test["LS Boost"] /= comp_table_test["Number of Samples"]

# compute delta error against baseline
comp_table_test["MC Boost"] -= comp_table_test["Baseline"]
comp_table_test["KMAcc"] -= comp_table_test["Baseline"]
comp_table_test["KMAcc+ISO"] -= comp_table_test["Baseline"]
comp_table_test["LS Boost"] -= comp_table_test["Baseline"]

# Add overall performance on the last row
comp_table_test.loc["Total"] = [comp_table_test['Number of Samples'].sum(), test_df["Baseline"].mean(), test_df["MC Boost"].mean() - test_df["Baseline"].mean(),
                                         test_df["KMAcc"].mean() - test_df["Baseline"].mean() ,test_df["KMAcc+ISO"].mean() - test_df["Baseline"].mean(), test_df["LS Boost"].mean() - test_df["Baseline"].mean()]


comp_table_test["Number of Samples"] = comp_table_test["Number of Samples"].astype(int)
columns = comp_table_test.columns.drop(["Number of Samples"])
comp_table_test[columns] = (comp_table_test[columns]).round(4)


# rename columns to Delta Error
comp_table_test = comp_table_test.rename(columns={"Baseline": "Baseline MSE", "MC Boost": u"MC Boost \u0394", 
                                                  "KMAcc": u"KMAcc \u0394", "KMAcc+ISO": u"KMAcc+ISO \u0394", 
                                                  "LS Boost": u"LS Boost \u0394"})


In [102]:
pd.options.display.multi_sparse = False

In [103]:
print("Mean-Squared Error on Test Data")
# apply color to the table, darker color means lower value, only to last 4 columns
vmin = comp_table_test[comp_table_test.columns[2:]].min().min()
vmax = 0
comp_table_test.style.background_gradient(cmap='Greens_r', subset=comp_table_test.columns[2:], axis = 1, vmin = vmin, vmax = vmax)

Mean-Squared Error on Test Data


,Number of Samples,Baseline MSE,MC Boost Δ,KMAcc Δ,KMAcc+ISO Δ,LS Boost Δ
"('Foreign Born', 'Female', 'Non-White')",819,0.143700,-0.007500,-0.001600,-0.002400,0.040700
"('Foreign Born', 'Female', 'White')",484,0.184000,-0.003300,0.000300,-0.000000,0.055600
"('Foreign Born', 'Male', 'Non-White')",411,0.175600,-0.003800,-0.001700,-0.005900,0.043400
"('Foreign Born', 'Male', 'White')",314,0.184400,-0.000800,-0.002100,-0.002900,0.064000
"('Native', 'Female', 'Non-White')",805,0.193500,-0.001000,-0.006600,-0.001800,0.093500
"('Native', 'Female', 'White')",2672,0.127200,-0.000700,-0.004300,-0.003800,0.038200
"('Native', 'Male', 'Non-White')",821,0.206600,-0.001700,-0.007300,-0.007300,0.090600
"('Native', 'Male', 'White')",1957,0.146700,-0.001200,-0.003500,-0.003200,0.041300
Total,8283,0.155600,-0.001900,-0.003900,-0.003500,0.052000


#### Multiaccuracy (Group Bias)

In [104]:
# Group attributes
group_attrs = ['NATIVITY', 'SEX','RAC1P']

# Test dataset with group attributes and error
test_df = pd.DataFrame(X_test, columns=features_df.columns)
test_df = test_df[group_attrs]

# Add a column of 1's dummy to calculate number of samples
test_df["Number of Samples"] = 1

test_df["Baseline"] = (baseline_test_prob-y_test)
test_df["MC Boost"] = (mcb_test_prob-y_test)
test_df["KMAcc"] = (kma_test_prob-y_test)
test_df["KMAcc+ISO"] = (kma_iso_test_prob-y_test)
test_df["LS Boost"] = (lsboost_test_prob-y_test)

In [105]:
comp_table_test = test_df.groupby(group_attrs).sum()

In [106]:
comp_table_test["Baseline"] /= comp_table_test["Number of Samples"] 
comp_table_test["MC Boost"] /= comp_table_test["Number of Samples"]
comp_table_test["KMAcc"] /= comp_table_test["Number of Samples"]
comp_table_test["KMAcc+ISO"] /= comp_table_test["Number of Samples"]
comp_table_test["LS Boost"] /= comp_table_test["Number of Samples"]

# take absolute value
comp_table_test = comp_table_test.abs()

# compute delta error against baseline
comp_table_test["MC Boost"] -= comp_table_test["Baseline"]
comp_table_test["KMAcc"] -= comp_table_test["Baseline"]
comp_table_test["KMAcc+ISO"] -= comp_table_test["Baseline"]
comp_table_test["LS Boost"] -= comp_table_test["Baseline"]

# # Add overall performance on the last row
comp_table_test.loc["Total"] = [comp_table_test['Number of Samples'].sum(), abs(test_df["Baseline"].mean()), abs(test_df["MC Boost"].mean()) - abs(test_df["Baseline"].mean()),
                                         abs(test_df["KMAcc"].mean()) - abs(test_df["Baseline"].mean()) ,abs(test_df["KMAcc+ISO"].mean()) - abs(test_df["Baseline"].mean()), abs(test_df["LS Boost"].mean()) - abs(test_df["Baseline"].mean())]

# # Add overall performance on the last row
# comp_table_test.loc["Total"] = [comp_table_test['Number of Samples'].sum(), abs(test_df["Baseline"].mean()), abs(test_df["MC Boost"].mean()) ,
#                                          abs(test_df["KMAcc"].mean())  , abs(test_df["KMAcc+ISO"].mean()), abs(test_df["LS Boost"].mean())]


comp_table_test["Number of Samples"] = comp_table_test["Number of Samples"].astype(int)
columns = comp_table_test.columns.drop(["Number of Samples"])
comp_table_test[columns] = (comp_table_test[columns]).round(4)


# rename columns to Delta Error
comp_table_test = comp_table_test.rename(columns={"Baseline": "Baseline Bias", "MC Boost": u"MC Boost \u0394", 
                                                  "KMAcc": u"KMAcc \u0394", "KMAcc+ISO": u"KMAcc+ISO \u0394", 
                                                  "LS Boost": u"LS Boost \u0394"})


In [107]:
print("Per-group Bias (Multiaccuracy) on Test Data")
print("|E_g(f - Y)|")
# apply color to the table, darker color means lower value, only to last 4 columns
vmin = comp_table_test[comp_table_test.columns[2:]].min().min()
vmax = 0
comp_table_test.style.background_gradient(cmap='Greens_r', subset=comp_table_test.columns[2:], axis = 1, vmin = vmin, vmax = vmax)

Per-group Bias (Multiaccuracy) on Test Data
|E_g(f - Y)|


,Number of Samples,Baseline Bias,MC Boost Δ,KMAcc Δ,KMAcc+ISO Δ,LS Boost Δ
"('Foreign Born', 'Female', 'Non-White')",819,0.040900,-0.007500,-0.007200,-0.012000,0.075100
"('Foreign Born', 'Female', 'White')",484,0.041400,0.006600,-0.003700,0.000800,0.148700
"('Foreign Born', 'Male', 'Non-White')",411,0.010700,-0.009300,-0.001400,-0.009400,0.145000
"('Foreign Born', 'Male', 'White')",314,0.014600,0.012300,0.006300,0.014300,0.157400
"('Native', 'Female', 'Non-White')",805,0.067000,-0.002900,-0.004000,0.007700,0.152900
"('Native', 'Female', 'White')",2672,0.021400,0.000900,-0.021200,-0.021100,0.095300
"('Native', 'Male', 'Non-White')",821,0.054800,0.001700,-0.002800,0.010500,0.132800
"('Native', 'Male', 'White')",1957,0.004600,-0.000200,0.004100,0.008700,0.110900
Total,8283,0.002300,0.001800,0.010100,0.015300,0.139400
